# ADNI Target Subject Selection for Image Downloads

This notebook follows the attached **ADNI Full-Version Image Download Workflow**. It starts from processed tau PET Study Files, identifies subjects with longitudinal usable tau PET scans, then finds nearby DWI/DTI and T1/MPRAGE image series for manual download from the ADNI LONI Images tab.

Main decisions encoded here:

- Tau PET defines the cohort.
- Tau scans must pass UC Berkeley QC (`qc_flag == 2`).
- Longitudinal tau must use the same tracer, because SUVRs should not be compared across tracers.
- DWI/DTI must be within `MATCH_WINDOW_DAYS` of baseline tau PET.
- T1/MPRAGE must be within `MATCH_WINDOW_DAYS` of both DWI/DTI and baseline tau PET.
- DWI/DTI and T1 images are ranked by date closeness, QC availability, DWI volume count, tau follow-up depth, and amyloid-study-file availability.

Generated files:

- `outputs/adni_image_download_targets.csv`: full ranked eligible list.
- `outputs/adni_image_download_targets_top20.csv`: short review list.
- `outputs/adni_primary_target_download_manifest.csv`: two required image rows for the top target.


In [11]:
from pathlib import Path
import csv
import re
from collections import defaultdict, Counter
from datetime import datetime

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "ADNI").exists() and (PROJECT_ROOT.parent / "ADNI").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

DATA_ROOT = PROJECT_ROOT / "ADNI"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

MATCH_WINDOW_DAYS = 365
TAU_PASS_VALUES = {"2"}  # UC Berkeley tau qc_flag: 2=Pass, 1=Partial pass, 0=Fail.
ALLOW_PARTIAL_TAU_QC = False

print(f"Project root: {PROJECT_ROOT}")
print(f"ADNI Study Files: {DATA_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")


Project root: /Users/nguyenlinhda/Library/CloudStorage/OneDrive-TheUniversityofMelbourne/Desktop/Individual-Data-Augmented-Connectome-
ADNI Study Files: /Users/nguyenlinhda/Library/CloudStorage/OneDrive-TheUniversityofMelbourne/Desktop/Individual-Data-Augmented-Connectome-/ADNI
Output directory: /Users/nguyenlinhda/Library/CloudStorage/OneDrive-TheUniversityofMelbourne/Desktop/Individual-Data-Augmented-Connectome-/output


In [12]:
def read_csv(relative_path):
    path = DATA_ROOT / relative_path
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def parse_date(value):
    value = (value or "").strip()
    if not value:
        return None
    value = value.split()[0]
    for fmt in ("%Y-%m-%d", "%m/%d/%Y", "%Y%m%d"):
        try:
            return datetime.strptime(value, fmt).date()
        except ValueError:
            pass
    return None


def days_between(a, b):
    return abs((a - b).days) if a and b else None


def as_int(value, default=0):
    try:
        return int(float(str(value).strip()))
    except Exception:
        return default


def as_float(value, default=None):
    try:
        return float(str(value).strip())
    except Exception:
        return default


def strdate(value):
    return value.isoformat() if hasattr(value, "isoformat") else ("" if value is None else str(value))


def diag_label(code):
    return {"1": "CN", "2": "MCI", "3": "AD"}.get((code or "").strip(), code or "")


## 1. Load Subject Linkage and Build the Longitudinal Tau Cohort

The processed UC Berkeley tau table is the cohort anchor. This cell keeps pass-only tau scans and requires at least two scan dates for the same `RID` and tracer.


In [13]:
roster = read_csv("Enrollment/ROSTER_03May2026.csv")
ptid_to_rid = {r["PTID"]: r["RID"] for r in roster if r.get("PTID") and r.get("RID")}
rid_to_ptid = {r["RID"]: r["PTID"] for r in roster if r.get("PTID") and r.get("RID")}

tau = read_csv("PET_Image_Analysis/UCBERKELEY_TAU_6MM_03May2026.csv")
if ALLOW_PARTIAL_TAU_QC:
    tau_pass_values = TAU_PASS_VALUES | {"1"}
else:
    tau_pass_values = TAU_PASS_VALUES

print("Tau rows:", len(tau))
print("Tau qc_flag distribution:", Counter(r.get("qc_flag", "") for r in tau))
print("Tau tracer distribution:", Counter(r.get("TRACER", "") for r in tau))

tau_groups = defaultdict(list)
for row in tau:
    scan_date = parse_date(row.get("SCANDATE"))
    if not scan_date:
        continue
    if row.get("qc_flag") not in tau_pass_values:
        continue
    tau_groups[(row.get("RID", ""), row.get("TRACER", ""))].append(row)

long_tau = []
for (rid, tracer), rows in tau_groups.items():
    by_date = {}
    for row in rows:
        scan_date = parse_date(row.get("SCANDATE"))
        if scan_date and scan_date not in by_date:
            by_date[scan_date] = row
    dates = sorted(by_date)
    if len(dates) < 2:
        continue
    values = [as_float(by_date[d].get("META_TEMPORAL_SUVR")) for d in dates]
    long_tau.append({
        "RID": rid,
        "PTID": by_date[dates[0]].get("PTID") or rid_to_ptid.get(rid, ""),
        "TRACER": tracer,
        "tau_n": len(dates),
        "baseline_tau_date": dates[0],
        "last_tau_date": dates[-1],
        "tau_followup_dates": ";".join(str(d) for d in dates[1:]),
        "tau_dates": dates,
        "baseline_meta_temporal_suvr": values[0],
        "last_meta_temporal_suvr": values[-1],
        "annualized_meta_temporal_suvr_change": (
            (values[-1] - values[0]) / ((dates[-1] - dates[0]).days / 365.25)
            if values[0] is not None and values[-1] is not None and dates[-1] > dates[0]
            else None
        ),
        "tau_loniuids": ";".join(by_date[d].get("LONIUID", "") for d in dates),
        "tau_qc_status": "UCBerkeley qc_flag=2 for all selected tau scans" if not ALLOW_PARTIAL_TAU_QC else "UCBerkeley qc_flag in {1,2}",
        "tau_duration_days": (dates[-1] - dates[0]).days,
    })

print("Pass-only same-tracer longitudinal tau groups:", len(long_tau))
print("Unique RIDs:", len({s["RID"] for s in long_tau}))


Tau rows: 2489
Tau qc_flag distribution: Counter({'2': 2311, '-1': 137, '1': 30, '-2': 9, '0': 2})
Tau tracer distribution: Counter({'FTP': 2280, 'MK6240': 139, 'PI2620': 70})
Pass-only same-tracer longitudinal tau groups: 541
Unique RIDs: 541


## 2. Identify DWI/DTI and T1/MPRAGE Image Series

The `MRIQC` table carries the LONI image, series, and study identifiers needed for manual image download. The DTI ROI tables and MAYO ADNI3 MRI QC table are used as confirmation/enrichment when available.


In [14]:
mriqc = read_csv("MR_Image_Quality/MRIQC_04May2026.csv")

dwi_re = re.compile(r"\b(DTI|DWI|HARDI|DTB)\b|diffus", re.I)
t1_desc_re = re.compile(r"MPRAGE|MP[- ]?RAGE|\bT1\b|IR[- ]?SPGR|\bSPGR\b|BRAVO|SAG.*3D", re.I)


def classify_mri(row):
    series_type = (row.get("SeriesType") or "").strip().lower()
    description = row.get("SeriesDescription") or ""
    text = f"{series_type} {description}"
    is_dwi = series_type == "dmri" or bool(dwi_re.search(text))
    is_t1 = series_type == "t1w" or bool(t1_desc_re.search(description))
    return is_dwi, is_t1


mri_by_rid = defaultdict(lambda: {"DWI": [], "T1": []})
for row in mriqc:
    rid = ptid_to_rid.get((row.get("ParticipantID") or "").strip())
    scan_date = parse_date(row.get("StudyDate"))
    if not rid or not scan_date:
        continue
    is_dwi, is_t1 = classify_mri(row)
    event = {
        "RID": rid,
        "PTID": row.get("ParticipantID", ""),
        "date": scan_date,
        "image_id": row.get("image_id", ""),
        "loni_image": row.get("LONIImage", "") or row.get("image_id", ""),
        "loni_series": row.get("LONISeries", ""),
        "loni_study": row.get("LONIStudy", ""),
        "series_type": row.get("SeriesType", ""),
        "description": row.get("SeriesDescription", ""),
        "field_strength": row.get("MagneticFieldStrength", ""),
        "manufacturer": row.get("ScannerManufacturer", ""),
        "model": row.get("ScannerModel", ""),
        "n_volumes": row.get("NumberVolumes", ""),
        "visit": row.get("VISCODE2", ""),
        "series_instance_uid": row.get("SeriesInstanceUID", ""),
    }
    if is_dwi:
        mri_by_rid[rid]["DWI"].append(event)
    if is_t1:
        mri_by_rid[rid]["T1"].append(event)

print("MRIQC rows:", len(mriqc))
print("Classified DWI/DTI series:", sum(len(v["DWI"]) for v in mri_by_rid.values()))
print("Classified T1/MPRAGE series:", sum(len(v["T1"]) for v in mri_by_rid.values()))


MRIQC rows: 91414
Classified DWI/DTI series: 5778
Classified T1/MPRAGE series: 25522


In [15]:
mayo = read_csv("MR_Image_Quality/MAYOADIRL_MRI_QUALITY_ADNI3_04May2026.csv")
mayo_by_image = {}
mayo_by_series = {}
for row in mayo:
    if row.get("LONI_IMAGE"):
        mayo_by_image[str(row["LONI_IMAGE"]).strip()] = row
    if row.get("LONI_SERIES"):
        mayo_by_series[str(row["LONI_SERIES"]).strip()] = row


def mayo_row(event):
    return (
        mayo_by_image.get(str(event.get("image_id", "")).strip())
        or mayo_by_image.get(str(event.get("loni_image", "")).strip())
        or mayo_by_series.get(str(event.get("loni_series", "")).strip())
    )


def mayo_status(event):
    match = mayo_row(event)
    if not match:
        return "not in MAYO ADNI3 QC table"
    fields = [
        "STUDY_OVERALLPASS",
        "STUDY_PROTOCOL_STATUS",
        "SERIES_QUALITY",
        "PROTOCOL_STATUS",
        "SERIES_SELECTED",
        "STUDY_RESCAN_REQUESTED",
    ]
    return "; ".join(f"{field}={match.get(field, '')}" for field in fields)


def mayo_passish(event):
    match = mayo_row(event)
    if not match:
        return True
    if match.get("STUDY_RESCAN_REQUESTED") == "TRUE":
        return False
    if match.get("STUDY_OVERALLPASS") not in ("", "1"):
        return False
    return True


dti_confirm_by_image = defaultdict(list)
for relative_path in [
    "MR_Image_Analysis/DTIROI_ROBUSTMEAN_04May2026.csv",
    "MR_Image_Analysis/DTIROI_MEAN_04May2026.csv",
]:
    for row in read_csv(relative_path):
        for col in ["IMAGEUID", "IMAGEUID_AP"]:
            uid = (row.get(col) or "").strip()
            if uid:
                dti_confirm_by_image[uid].append({
                    "table": Path(relative_path).name,
                    "status": row.get("STATUS", ""),
                    "qc": row.get("QC", ""),
                    "volumes": row.get("VOLUMES", ""),
                    "volumes_ap": row.get("VOLUMES_AP", ""),
                    "distortion_correction": row.get("DISTORTION_CORRECTION", ""),
                    "examdate": row.get("EXAMDATE", ""),
                })

for row in read_csv("MR_Image_Analysis/ADNI_DTIROI_V1_04May2026.csv"):
    for col in ["LONIUID_1", "LONIUID_2", "LONIUID_3", "LONIUID_4"]:
        uid = (row.get(col) or "").strip().lstrip("I")
        if uid:
            dti_confirm_by_image[uid].append({
                "table": "ADNI_DTIROI_V1_04May2026.csv",
                "status": row.get("STATUS", ""),
                "qc": "",
                "volumes": "",
                "volumes_ap": "",
                "distortion_correction": "",
                "examdate": row.get("EXAMDATE", ""),
            })


def dti_status(event):
    rows = (
        dti_confirm_by_image.get(str(event.get("image_id", "")).strip())
        or dti_confirm_by_image.get(str(event.get("loni_image", "")).strip())
    )
    if not rows:
        return "not found in DTI ROI tables"
    parts = []
    for row in rows[:3]:
        extra = []
        if row.get("volumes"):
            extra.append(f"volumes={row['volumes']}")
        if row.get("volumes_ap"):
            extra.append(f"AP={row['volumes_ap']}")
        if row.get("distortion_correction"):
            extra.append(f"distortion={row['distortion_correction']}")
        if row.get("qc"):
            extra.append(f"QC={row['qc']}")
        parts.append(
            f"{row['table']} status={row.get('status', '')}"
            + (f" ({', '.join(extra)})" if extra else "")
        )
    return "; ".join(parts)


## 3. Add Covariates and Optional Amyloid Availability

These fields do not drive the image download directly, but they help decide whether the subject is useful for the full model.


In [16]:
amy_by_rid = defaultdict(list)
for row in read_csv("PET_Image_Analysis/UCBERKELEY_AMY_6MM_03May2026.csv"):
    scan_date = parse_date(row.get("SCANDATE"))
    if scan_date and row.get("qc_flag", "2") not in {"0", "-2"}:
        amy_by_rid[row.get("RID", "")].append(scan_date)


dx_by_rid = defaultdict(list)
for row in read_csv("Diagnosis/DXSUM_03May2026.csv"):
    exam_date = parse_date(row.get("EXAMDATE"))
    if exam_date:
        dx_by_rid[row.get("RID", "")].append((exam_date, row))

ptdemog_by_rid = {}
for row in read_csv("Subject_Demographics/PTDEMOG_03May2026.csv"):
    rid = row.get("RID")
    if rid and rid not in ptdemog_by_rid:
        ptdemog_by_rid[rid] = row

apoe_by_rid = {}
for row in read_csv("Genetic_APOE/APOERES_03May2026.csv"):
    rid = row.get("RID")
    if rid and rid not in apoe_by_rid:
        apoe_by_rid[rid] = row


def nearest_dx(rid, target_date):
    rows = dx_by_rid.get(rid, [])
    if not rows or not target_date:
        return None
    return min(rows, key=lambda item: abs((item[0] - target_date).days))[1]

print("Subjects with processed amyloid rows:", len(amy_by_rid))
print("Subjects with diagnosis rows:", len(dx_by_rid))
print("Subjects with APOE rows:", len(apoe_by_rid))


Subjects with processed amyloid rows: 2229
Subjects with diagnosis rows: 3685
Subjects with APOE rows: 3208


## 4. Select the Best DWI/T1 Pair per Longitudinal Tau Subject

Ranking is deliberately conservative. It prefers a same-day DWI/T1 pair, close to baseline tau, with complete DTI ROI confirmation and richer longitudinal tau follow-up.


In [17]:
def dwi_score(event, baseline_tau_date):
    delta = days_between(event["date"], baseline_tau_date)
    description = (event.get("description") or "").upper()
    return (
        delta if delta is not None else 10**9,
        0 if mayo_passish(event) else 1,
        -as_int(event.get("n_volumes")),
        0 if any(term in description for term in ["DTI", "DWI", "DIFF"]) else 1,
        str(event.get("image_id", "")),
    )


def t1_score(event, dwi_date):
    delta = days_between(event["date"], dwi_date)
    description = (event.get("description") or "").upper()
    field_strength = as_float(event.get("field_strength"), 0) or 0
    return (
        delta if delta is not None else 10**9,
        0 if mayo_passish(event) else 1,
        0 if "MPRAGE" in description or "MP-RAGE" in description else 1,
        0 if "REPEAT" not in description else 1,
        -field_strength,
        str(event.get("image_id", "")),
    )


def choose_pair(rid, baseline_tau_date, window_days=MATCH_WINDOW_DAYS):
    dwi_candidates = [
        event
        for event in mri_by_rid.get(rid, {}).get("DWI", [])
        if days_between(event["date"], baseline_tau_date) is not None
        and days_between(event["date"], baseline_tau_date) <= window_days
        and mayo_passish(event)
    ]
    if not dwi_candidates:
        return None, None

    dwi = sorted(dwi_candidates, key=lambda event: dwi_score(event, baseline_tau_date))[0]
    t1_candidates = [
        event
        for event in mri_by_rid.get(rid, {}).get("T1", [])
        if days_between(event["date"], dwi["date"]) is not None
        and days_between(event["date"], dwi["date"]) <= window_days
        and days_between(event["date"], baseline_tau_date) <= window_days
        and mayo_passish(event)
    ]
    if not t1_candidates:
        return dwi, None

    t1 = sorted(t1_candidates, key=lambda event: t1_score(event, dwi["date"]))[0]
    return dwi, t1


In [18]:
candidate_rows = []
for subject in long_tau:
    rid = subject["RID"]
    baseline_tau_date = subject["baseline_tau_date"]
    dwi, t1 = choose_pair(rid, baseline_tau_date)
    if not dwi or not t1:
        continue

    dx = nearest_dx(rid, baseline_tau_date)
    demographics = ptdemog_by_rid.get(rid, {})
    apoe = apoe_by_rid.get(rid, {})
    amyloid_dates = sorted(set(amy_by_rid.get(rid, [])))

    row = {key: value for key, value in subject.items() if key != "tau_dates"}
    row.update({
        "DWI_image_id": dwi.get("image_id", ""),
        "DWI_series_id": dwi.get("loni_series", ""),
        "DWI_study_id": dwi.get("loni_study", ""),
        "DWI_date": dwi["date"],
        "DWI_days_from_baseline_tau": (dwi["date"] - baseline_tau_date).days,
        "DWI_description": dwi.get("description", ""),
        "DWI_series_type": dwi.get("series_type", ""),
        "DWI_n_volumes": dwi.get("n_volumes", ""),
        "DWI_field_strength": dwi.get("field_strength", ""),
        "DWI_manufacturer": dwi.get("manufacturer", ""),
        "DWI_dtiroi_status": dti_status(dwi),
        "DWI_mri_qc_status": mayo_status(dwi),
        "T1_image_id": t1.get("image_id", ""),
        "T1_series_id": t1.get("loni_series", ""),
        "T1_study_id": t1.get("loni_study", ""),
        "T1_date": t1["date"],
        "T1_days_from_DWI": (t1["date"] - dwi["date"]).days,
        "T1_days_from_baseline_tau": (t1["date"] - baseline_tau_date).days,
        "T1_description": t1.get("description", ""),
        "T1_series_type": t1.get("series_type", ""),
        "T1_field_strength": t1.get("field_strength", ""),
        "T1_manufacturer": t1.get("manufacturer", ""),
        "T1_mri_qc_status": mayo_status(t1),
        "dx_nearest_tau": diag_label(dx.get("DIAGNOSIS", "") if dx else ""),
        "dx_nearest_tau_date": dx.get("EXAMDATE", "") if dx else "",
        "sex_code": demographics.get("PTGENDER", ""),
        "education_years": demographics.get("PTEDUCAT", ""),
        "apoe_genotype": apoe.get("GENOTYPE", ""),
        "amyloid_scan_count": len(amyloid_dates),
        "amyloid_dates": ";".join(str(d) for d in amyloid_dates),
        "raw_tau_pet_needed": "No",
        "raw_amyloid_pet_needed": "No",
        "download_priority": "High",
        "manual_download_note": "Download DWI/DTI and T1/MPRAGE rows by LONI image ID or series ID from ADNI Search & Download > Images.",
    })

    dwi_abs_days = abs(row["DWI_days_from_baseline_tau"])
    t1_abs_days = abs(row["T1_days_from_DWI"])
    row["_score"] = (
        0 if dwi_abs_days <= 180 else 1,
        0 if t1_abs_days == 0 else (1 if t1_abs_days <= 30 else 2),
        -as_int(row.get("DWI_n_volumes")),
        -row["tau_n"],
        -row["tau_duration_days"],
        -len(amyloid_dates),
        dwi_abs_days,
        t1_abs_days,
        as_int(rid, 999999),
    )
    candidate_rows.append(row)

candidate_rows = sorted(candidate_rows, key=lambda row: row["_score"])
for rank, row in enumerate(candidate_rows, start=1):
    row["rank"] = rank

print("Eligible subjects with longitudinal tau + DWI + T1:", len(candidate_rows))
print("Top RID:", candidate_rows[0]["RID"], candidate_rows[0]["PTID"] if candidate_rows else "")


Eligible subjects with longitudinal tau + DWI + T1: 501
Top RID: 5265 007_S_5265


## 5. Export Download Lists

Use the primary manifest for the immediate manual LONI download. Use the top-20 and full list if you want to switch to an MCI/AD-only subject or download a small pilot batch.


In [19]:
fieldnames = [
    "rank", "RID", "PTID", "TRACER", "tau_n", "baseline_tau_date", "last_tau_date",
    "tau_followup_dates", "tau_duration_days", "tau_loniuids", "tau_qc_status",
    "baseline_meta_temporal_suvr", "last_meta_temporal_suvr", "annualized_meta_temporal_suvr_change",
    "DWI_image_id", "DWI_series_id", "DWI_study_id", "DWI_date", "DWI_days_from_baseline_tau",
    "DWI_description", "DWI_series_type", "DWI_n_volumes", "DWI_field_strength", "DWI_manufacturer",
    "DWI_dtiroi_status", "DWI_mri_qc_status",
    "T1_image_id", "T1_series_id", "T1_study_id", "T1_date", "T1_days_from_DWI", "T1_days_from_baseline_tau",
    "T1_description", "T1_series_type", "T1_field_strength", "T1_manufacturer", "T1_mri_qc_status",
    "dx_nearest_tau", "dx_nearest_tau_date", "sex_code", "education_years", "apoe_genotype",
    "amyloid_scan_count", "amyloid_dates", "raw_tau_pet_needed", "raw_amyloid_pet_needed",
    "download_priority", "manual_download_note",
]


def write_candidate_csv(path, rows):
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: strdate(row.get(key, "")) for key in fieldnames})

write_candidate_csv(OUTPUT_DIR / "adni_image_download_targets.csv", candidate_rows)
write_candidate_csv(OUTPUT_DIR / "adni_image_download_targets_top20.csv", candidate_rows[:20])

primary = candidate_rows[0]
manifest = []
for modality in ["DWI", "T1"]:
    manifest.append({
        "RID": primary["RID"],
        "PTID": primary["PTID"],
        "modality": modality,
        "image_id": primary[f"{modality}_image_id"],
        "series_id": primary[f"{modality}_series_id"],
        "study_id": primary[f"{modality}_study_id"],
        "scan_date": strdate(primary[f"{modality}_date"]),
        "description": primary[f"{modality}_description"],
        "field_strength": primary.get(f"{modality}_field_strength", ""),
        "download_priority": "Required / High",
        "qc_status": primary.get(f"{modality}_mri_qc_status", ""),
        "note": "Search this image_id or series_id in ADNI LONI Images tab.",
    })

with (OUTPUT_DIR / "adni_primary_target_download_manifest.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(manifest[0].keys()))
    writer.writeheader()
    writer.writerows(manifest)

print("Wrote:")
print(OUTPUT_DIR / "adni_image_download_targets.csv")
print(OUTPUT_DIR / "adni_image_download_targets_top20.csv")
print(OUTPUT_DIR / "adni_primary_target_download_manifest.csv")


Wrote:
/Users/nguyenlinhda/Library/CloudStorage/OneDrive-TheUniversityofMelbourne/Desktop/Individual-Data-Augmented-Connectome-/output/adni_image_download_targets.csv
/Users/nguyenlinhda/Library/CloudStorage/OneDrive-TheUniversityofMelbourne/Desktop/Individual-Data-Augmented-Connectome-/output/adni_image_download_targets_top20.csv
/Users/nguyenlinhda/Library/CloudStorage/OneDrive-TheUniversityofMelbourne/Desktop/Individual-Data-Augmented-Connectome-/output/adni_primary_target_download_manifest.csv


## 6. Recommended Target and Alternatives

The top-ranked subject is the best first manual download under the default rules. If you want a symptomatic-only target, review the MCI/AD alternatives printed below.


In [20]:
def show_row(row):
    keys = [
        "rank", "RID", "PTID", "dx_nearest_tau", "TRACER", "tau_n", "baseline_tau_date", "last_tau_date",
        "tau_duration_days", "baseline_meta_temporal_suvr", "annualized_meta_temporal_suvr_change",
        "DWI_image_id", "DWI_series_id", "DWI_date", "DWI_days_from_baseline_tau", "DWI_description", "DWI_n_volumes",
        "T1_image_id", "T1_series_id", "T1_date", "T1_days_from_DWI", "T1_description",
        "apoe_genotype", "amyloid_scan_count",
    ]
    for key in keys:
        print(f"{key}: {strdate(row.get(key, ''))}")

print("Primary target")
print("--------------")
show_row(candidate_rows[0])

print("\nTop 10 compact table")
print("--------------------")
compact_fields = ["rank", "RID", "PTID", "dx_nearest_tau", "tau_n", "baseline_tau_date", "DWI_image_id", "DWI_date", "T1_image_id", "T1_date", "baseline_meta_temporal_suvr", "annualized_meta_temporal_suvr_change"]
print("	".join(compact_fields))
for row in candidate_rows[:10]:
    print("	".join(strdate(row.get(field, "")) for field in compact_fields))

print("\nBest MCI/AD alternatives")
print("------------------------")
for row in [r for r in candidate_rows if r.get("dx_nearest_tau") in {"MCI", "AD"}][:10]:
    print("	".join(strdate(row.get(field, "")) for field in compact_fields))


Primary target
--------------
rank: 1
RID: 5265
PTID: 007_S_5265
dx_nearest_tau: CN
TRACER: FTP
tau_n: 5
baseline_tau_date: 2017-09-20
last_tau_date: 2024-11-12
tau_duration_days: 2610
baseline_meta_temporal_suvr: 1.556
annualized_meta_temporal_suvr_change: 0.10705603448275865
DWI_image_id: 905404
DWI_series_id: 609776
DWI_date: 2017-09-18
DWI_days_from_baseline_tau: -2
DWI_description: Axial MB DTI
DWI_n_volumes: 127
T1_image_id: 905391
T1_series_id: 609766
T1_date: 2017-09-18
T1_days_from_DWI: 0
T1_description: Accelerated Sagittal MPRAGE
apoe_genotype: 3/3
amyloid_scan_count: 6

Top 10 compact table
--------------------
rank	RID	PTID	dx_nearest_tau	tau_n	baseline_tau_date	DWI_image_id	DWI_date	T1_image_id	T1_date	baseline_meta_temporal_suvr	annualized_meta_temporal_suvr_change
1	5265	007_S_5265	CN	5	2017-09-20	905404	2017-09-18	905391	2017-09-18	1.556	0.10705603448275865
2	4488	007_S_4488	CN	5	2017-09-13	902672	2017-09-12	902659	2017-09-12	1.097	0.0008573943661971838
3	6333	941_S_63

## Manual LONI Download Notes

For the primary target, download only the required DWI/DTI and T1/MPRAGE image series first:

1. Open ADNI LONI Search & Download -> Images.
2. Search by `PTID`/subject ID or directly by the `image_id` / `series_id` in `outputs/adni_primary_target_download_manifest.csv`.
3. Download the DWI/DTI series and T1/MPRAGE series listed in the manifest.
4. Keep raw downloads separate from derived outputs, for example `ADNI_images/raw_downloads/RID_5265/DWI/` and `ADNI_images/raw_downloads/RID_5265/T1/`.
5. Raw tau and amyloid PET are marked `No` for the first pass because the UC Berkeley regional tables are already available.
